# Plume API Quick Start

A step-by-step walkthrough of the minimum requirements to get your first Plume API request up and running.

| Step | Action | Endpoint |
|------|--------|----------|
| 1 | Generate an Access Token | OAuth 2.0 Token URL |
| 2 | Configure connection details | — |
| 3 | Create a User | `POST /v1/users` |
| 4 | Get the User's Location | `GET /v1/locations/{id}` |
| 5 | Assign internal identifiers | `PATCH /v1/users/{id}` · `PATCH /v1/locations/{id}` |

Run cells **top to bottom**. Each cell's output feeds into the next step automatically via shared variables.

## Setup

Install the `requests` library if you haven't already, then import everything needed.

In [ ]:
%pip install requests --quiet

In [ ]:
import os
import json
import requests
from IPython.display import display, JSON

---
## Step 1 — Generate an Access Token

Plume APIs use **OAuth 2.0 Client Credentials** (machine-to-machine). After creating an API Token in
the Plume Portal you receive three values needed to mint a short-lived JWT bearer token:

- **Authorization Token URL** — the OAuth token endpoint for your deployment
- **Authorization Header** — `Basic <base64(clientId:clientSecret)>` (shown **once** in the Portal — store it safely)
- **Scope** — controls which APIs the token can access

**Best practice:** Load secrets from environment variables rather than hard-coding them.
Set `AUTH_TOKEN_URL`, `AUTH_HEADER`, and `AUTH_SCOPE` in your shell before launching Jupyter.

> ⚠️ **Reuse access tokens until they expire** — do not mint a new token per API call.

In [ ]:
# ── Credentials ────────────────────────────────────────────────────────────────
# Load from environment variables (preferred) or paste values here for quick testing.

AUTH_TOKEN_URL = os.environ.get("AUTH_TOKEN_URL", "https://yourco.okta.com/oauth2/default/v1/token")
AUTH_HEADER    = os.environ.get("AUTH_HEADER",    "Basic dXNlcjpwYXNzd29yZA==")
AUTH_SCOPE     = os.environ.get("AUTH_SCOPE",     "partnerId:6971f4934016be004a041191 role:partnerIdAdmin")

# ── Mint access token (OAuth 2.0 Client Credentials) ──────────────────────────
response = requests.post(
    AUTH_TOKEN_URL,
    headers={
        "Authorization": AUTH_HEADER,
        "Content-Type": "application/x-www-form-urlencoded",
    },
    data={
        "grant_type": "client_credentials",
        "scope": AUTH_SCOPE,
    },
    timeout=10,
)
response.raise_for_status()

token_data   = response.json()
ACCESS_TOKEN = token_data["access_token"]

print(f"✅ Access token minted. Expires in {token_data.get('expires_in', '?')} seconds.")
print(f"   Token (truncated): {ACCESS_TOKEN[:60]}…")

**Expected response structure:**

```json
{
  "token_type": "Bearer",
  "expires_in": 7200,
  "access_token": "eyJ…",
  "scope": "partnerId:<your-partner-id> role:partnerIdAdmin"
}
```

---
## Step 2 — Set Connection Details

Set the API base URL and your Partner ID. These are used as a prefix / body field in every subsequent request.

In [ ]:
API_URL    = os.environ.get("API_URL",    "https://api.usw2.prod.plume.tech")
PARTNER_ID = os.environ.get("PARTNER_ID", "6971f4934016be004a041191")

# Convenience: build an Authorization header dict reused in every call
AUTH = {"Authorization": f"Bearer {ACCESS_TOKEN}"}

print(f"API_URL    : {API_URL}")
print(f"PARTNER_ID : {PARTNER_ID}")

---
## Step 3 — Create a User

`POST /v1/users` creates a new customer. The request body requires **name**, **email**, and **partnerId**.

> **Note:** Creating a User also automatically provisions a default Location (`"Home"`, type `RESIDENTIAL`).
> A User cannot exist without at least one Location. The `locationId` is returned in the response
> and is used in Steps 4 and 5.

In [ ]:
create_user_payload = {
    "name":      "Jane Doe",
    "email":     "jdoe369@example.com",
    "partnerId": PARTNER_ID,
}

response = requests.post(
    f"{API_URL}/v1/users",
    headers={**AUTH, "Content-Type": "application/json; charset=utf-8"},
    json=create_user_payload,
    timeout=10,
)
response.raise_for_status()

created_user = response.json()

# Extract IDs for use in later steps
USER_ID     = created_user["id"]
LOCATION_ID = created_user["location"]["id"]

print(f"✅ User created.")
print(f"   userId     : {USER_ID}")
print(f"   locationId : {LOCATION_ID}")
print()
display(JSON(created_user))

**Expected response:**

```json
{
  "location": {
    "name": "Home",
    "type": "RESIDENTIAL",
    "id": "69b8ec866e816b004ae092a2"
  },
  "name": "Jane Doe",
  "email": "jdoe369@example.com",
  "partnerId": "6971f4934016be004a041191",
  "accountId": "ad3c7cfd-3261-4cdb-950d-8d5d9250fdc9",
  "id": "69b8ec866e816b004ae092a1",
  "createdAt": "2026-03-17T05:54:14.152Z"
}
```

---
## Step 4 — Get the User's Location

`GET /v1/locations/{locationId}` returns the full Location record.

We use the `LOCATION_ID` extracted from the Step 3 response.

In [ ]:
response = requests.get(
    f"{API_URL}/v1/locations/{LOCATION_ID}",
    headers=AUTH,
    timeout=10,
)
response.raise_for_status()

location_data = response.json()

print(f"✅ Location retrieved.")
display(JSON(location_data))

**Expected response:**

```json
{
  "name": "Home",
  "createdAt": "2026-03-17T05:54:14.167Z",
  "type": "RESIDENTIAL",
  "id": "69b8ec866e816b004ae092a2",
  "partnerId": "6971f4934016be004a041191",
  "userId": "69b8ec866e816b004ae092a1"
}
```

---
## Step 5 — Assign Internal Identifiers *(optional)*

Link Plume records to your own CRM or billing system using two identifiers:

- **`accountId`** on the **User** — your customer's account reference (e.g., a UUID from your CRM)
- **`serviceId`** on the **Location** — the service contract reference (e.g., a subscription ID)

These make it easy to look up Plume data by your own IDs and vice versa.
See [accountId vs serviceId](../core-concepts/entity-hierarchy.md#accountid-vs-serviceid) for details.

### 5a — Update `accountId` on User

`PATCH /v1/users/{userId}` with an `accountId` field links the User to your CRM.

In [ ]:
YOUR_ACCOUNT_ID = "9eec69a9-5e27-4110-bdf3-146ed14e1632"  # replace with your CRM account reference

response = requests.patch(
    f"{API_URL}/v1/users/{USER_ID}",
    headers={**AUTH, "Content-Type": "application/json; charset=utf-8"},
    json={"accountId": YOUR_ACCOUNT_ID},
    timeout=10,
)
response.raise_for_status()

updated_user = response.json()

print(f"✅ accountId updated on User {USER_ID}")
display(JSON(updated_user))

**Expected response:**

```json
{
  "name": "Jane Doe",
  "email": "jdoe369@example.com",
  "partnerId": "6971f4934016be004a041191",
  "accountId": "9eec69a9-5e27-4110-bdf3-146ed14e1632",
  "id": "69b8ec866e816b004ae092a1",
  "createdAt": "2026-03-17T05:54:14.152Z"
}
```

### 5b — Update `serviceId` on Location

`PATCH /v1/locations/{locationId}` with a `serviceId` (and optionally `name` and `type`) links
the Location to a service contract in your CRM.

In [ ]:
YOUR_SERVICE_ID = "92283fcf-abbe-4580-ab0f-82a99ebda411"  # replace with your contract/subscription ID

response = requests.patch(
    f"{API_URL}/v1/locations/{LOCATION_ID}",
    headers={**AUTH, "Content-Type": "application/json; charset=utf-8"},
    json={
        "name":      "CentralParkApartment",
        "type":      "RESIDENTIAL",
        "serviceId": YOUR_SERVICE_ID,
    },
    timeout=10,
)
response.raise_for_status()

updated_location = response.json()

print(f"✅ serviceId updated on Location {LOCATION_ID}")
display(JSON(updated_location))

**Expected response:**

```json
{
  "name": "CentralParkApartment",
  "createdAt": "2026-03-17T05:54:14.167Z",
  "type": "RESIDENTIAL",
  "id": "69b8ec866e816b004ae092a2",
  "partnerId": "6971f4934016be004a041191",
  "userId": "69b8ec866e816b004ae092a1",
  "serviceId": "92283fcf-abbe-4580-ab0f-82a99ebda411"
}
```

---
## 🎉 Summary

With these steps complete you have successfully:

1. **Authenticated** via OAuth 2.0 Client Credentials and minted a JWT bearer token
2. **Created a User** with an auto-provisioned Location
3. **Retrieved the Location** record using the `locationId` from the create response
4. **Linked your CRM identifiers** — `accountId` on the User, `serviceId` on the Location

**Next steps:** explore the full API reference or read the
[entity hierarchy guide](../core-concepts/entity-hierarchy.md) to understand how
Users, Locations, and Nodes relate to each other.